In [1]:
# ==========================================
# CELL 1: SETUP REPOSITORY & DEPENDENCIES
# ==========================================
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

# 1. Clone project repository
!git clone https://github.com/Sabtain-Dev/STT-Whisper-Pashto.git
%cd STT-Whisper-Pashto

# 2. Install system-level audio tools
!apt-get update && apt-get install -y ffmpeg

# 3. Clean up requirements.txt by removing non-existent pip packages (e.g., localtunnel)
!sed -i '/localtunnel/d' requirements.txt

# 4. Install Python dependencies
!pip install -r requirements.txt
!pip install pydantic-settings jiwer python-multipart pyngrok -q

CUDA Available: True
Device Name: Tesla T4
Cloning into 'STT-Whisper-Pashto'...
remote: Enumerating objects: 203, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 203 (delta 12), reused 8 (delta 6), pack-reused 186 (from 1)
Receiving objects: 100% (203/203), 253.23 KiB | 11.51 MiB/s, done.
Resolving deltas: 100% (91/91), done.
/content/STT-Whisper-Pashto
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [101 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,845 kB

In [2]:
# ==========================================
# CELL 2: START FASTAPI BACKEND (GPU)
# ==========================================
import os
import time
from pyngrok import ngrok

# Configure ngrok Auth Token
NGROK_TOKEN = "3DlPbWM8eqOtqxCaw2x2eAvzbtA_5cRStnG1J42yBktkD1iLv"  # Replace with your token if needed
ngrok.set_auth_token(NGROK_TOKEN)

# Ensure PyTorch utilizes GPU inside Uvicorn subprocess
os.environ["PYTHONPATH"] = "."
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Kill any existing processes on port 8000
!fuser -k 8000/tcp || true

# Start FastAPI backend in background
print("--- STARTING FASTAPI BACKEND ON GPU ---")
os.system("nohup python -m uvicorn api.main:app --host 127.0.0.1 --port 8000 > fastapi.log 2>&1 &")
time.sleep(7)  # Pause to let Whisper load into GPU VRAM

# Expose FastAPI via ngrok
public_url_fastapi = ngrok.connect(8000)
print(f"FastAPI Public Gateway URL: {public_url_fastapi}")

# Store URL in environment for Streamlit cell
os.environ["FASTAPI_PUBLIC_URL"] = str(public_url_fastapi.public_url if hasattr(public_url_fastapi, 'public_url') else public_url_fastapi)

--- STARTING FASTAPI BACKEND ON GPU ---
FastAPI Public Gateway URL: NgrokTunnel: "https://previous-yiddish-skimming.ngrok-free.dev" -> "http://localhost:8000"


In [3]:
# ==========================================
# CELL 3: START STREAMLIT FRONTEND
# ==========================================
import os
import time
from pyngrok import ngrok

BACKEND_GATEWAY_URL = os.environ.get("FASTAPI_PUBLIC_URL")

if not BACKEND_GATEWAY_URL:
    raise ValueError("FASTAPI_PUBLIC_URL is not set. Run Cell 2 first.")

# Configure API URL for Streamlit frontend
os.environ["PASHTO_API_URL"] = f"{BACKEND_GATEWAY_URL}/api/v1"

# Kill any existing processes on port 8501
!fuser -k 8501/tcp || true

# Start Streamlit engine in background
print("--- STARTING STREAMLIT FRONTEND UI ---")
os.system("nohup python -m streamlit run app/app.py --server.port 8501 > streamlit.log 2>&1 &")
time.sleep(5)

# Expose Streamlit via ngrok
public_url_streamlit = ngrok.connect(8501)
print(f"\n=======================================================")
print(f"Streamlit App Public Link: {public_url_streamlit}")
print(f"=======================================================")

--- STARTING STREAMLIT FRONTEND UI ---

Streamlit App Public Link: NgrokTunnel: "https://previous-yiddish-skimming.ngrok-free.dev" -> "http://localhost:8501"


In [4]:
# ==========================================
# CELL 4: DEBUGGING & LOGS CHECKER
# ==========================================
# Run this cell if either server fails to respond.

print("=== FASTAPI LOGS (LAST 20 LINES) ===")
!tail -n 20 fastapi.log

print("\n=== STREAMLIT LOGS (LAST 20 LINES) ===")
!tail -n 20 streamlit.log

=== FASTAPI LOGS (LAST 20 LINES) ===
INFO:     Started server process [2492]
INFO:     Waiting for application startup.
[2026-07-21 11:18:07] [INFO] [pashto_asr:main.py:10] ── ==========================================================
[2026-07-21 11:18:07] [INFO] [pashto_asr:main.py:11] ──  Starting System Initialization Suite for: Pashto Whisper STT API
[2026-07-21 11:18:07] [INFO] [pashto_asr:main.py:12] ──  API Routing Space Context URL: /api/v1
[2026-07-21 11:18:07] [INFO] [pashto_asr:main.py:13] ── ==========================================================
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)

=== STREAMLIT LOGS (LAST 20 LINES) ===


2026-07-21 11:18:09.470 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.247.36.192:8501

